# 03. Feature Engineering & Preprocessing
**Objective:** Transform the cleaned dataset (`renewable_clean.csv`) into leakage-free, regression-ready training and testing matrices using chronological splitting and domain-driven feature engineering.

### Pipeline Checklist:
- [ ] Load verified clean dataset (`data/processed/renewable_clean.csv`)
- [ ] Prevent Target Leakage by dropping ex-post energy sub-components
- [ ] Engineer continuous time-trend index (`t`)
- [ ] Perform chronological Train-Test Split (`1973–2018` vs `2019–2023`)
- [ ] Encode `Sector`, `Month`, and `Sector x t` interaction terms (avoiding Dummy Variable Trap)
- [ ] Scale continuous features (`fit` on train, `transform` on test) & Export

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [22]:
CLEAN_PATH = Path('../data/processed/renewable_clean.csv')
PROCESSED_DIR = Path('../data/processed')

In [23]:
df = pd.read_csv(CLEAN_PATH)
print(f"Loaded clean dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(5)

Loaded clean dataset: 2,448 rows x 16 columns


,Year,Month,Sector,Geothermal Energy,Solar Energy,Wind Energy,Wood Energy,Waste Energy,"Fuel Ethanol, Excluding Denaturant",Biomass Losses and Co-products,Biomass Energy,Total Renewable Energy,Renewable Diesel Fuel,Other Biofuels,Biodiesel,hydro
0,1973,1,Commercial,0.00,0.0,0.0,0.570,0.000,0.0,0.0,0.570,0.570,0.0,0.0,0.0,0.000
1,1973,1,Electric Power,0.49,0.0,0.0,0.054,0.157,0.0,0.0,0.211,89.223,0.0,0.0,0.0,88.522
2,1973,1,Industrial,0.00,0.0,0.0,98.933,0.000,0.0,0.0,98.933,99.973,0.0,0.0,0.0,1.040
3,1973,1,Residential,0.00,0.0,0.0,30.074,0.000,0.0,0.0,0.000,30.074,0.0,0.0,0.0,0.000
4,1973,2,Commercial,0.00,0.0,0.0,0.515,0.000,0.0,0.0,0.515,0.515,0.0,0.0,0.0,0.000


### 1. Target Leakage Prevention
**Objective:** Retain only *ex-ante* known features (`Year`, `Month`, `Sector`) and our target (`Total Renewable Energy`). All raw energy sub-components (`hydro`, `Solar Energy`, `Wind Energy`, etc.) are dropped because they linearly sum up to the target and are unavailable when forecasting future periods.

In [24]:
keep_cols = ['Year','Month','Sector','Total Renewable Energy']
dropped_cols = [col for col in df.columns if col not in keep_cols]

df_feat = df[keep_cols].copy()
print(f"Dropped {len(dropped_cols)} sub-component columns to prevent Target Leakage:")
print(dropped_cols)
print(f"\nRemaining shape: {df_feat.shape[0]:,} rows x {df_feat.shape[1]} columns")
df_feat.head()

Dropped 12 sub-component columns to prevent Target Leakage:
['Geothermal Energy', 'Solar Energy', 'Wind Energy', 'Wood Energy', 'Waste Energy', 'Fuel Ethanol, Excluding Denaturant', 'Biomass Losses and Co-products', 'Biomass Energy', 'Renewable Diesel Fuel', 'Other Biofuels', 'Biodiesel', 'hydro']

Remaining shape: 2,448 rows x 4 columns


,Year,Month,Sector,Total Renewable Energy
0,1973,1,Commercial,0.570
1,1973,1,Electric Power,89.223
2,1973,1,Industrial,99.973
3,1973,1,Residential,30.074
4,1973,2,Commercial,0.515


### 2. Engineer Continuous Time Index (`t`)
**Objective:** Create a zero-indexed continuous monthly counter (`t` from `0` to `611`) so the regression model captures smooth month-over-month long-term growth from a meaningful baseline (`t = 0` at January 1973).

In [25]:
start_year = df_feat['Year'].min()
df_feat['t'] = (df_feat['Year'] - start_year) * 12 + (df_feat['Month'] - 1)
print(f"Time index 't' range: {df_feat['t'].min()} to {df_feat['t'].max()} ({df_feat['t'].nunique()} unique months)")
df_feat[['Year', 'Month', 'Sector', 't', 'Total Renewable Energy']].head(8)

Time index 't' range: 0 to 611 (612 unique months)


,Year,Month,Sector,t,Total Renewable Energy
0,1973,1,Commercial,0,0.570
1,1973,1,Electric Power,0,89.223
2,1973,1,Industrial,0,99.973
3,1973,1,Residential,0,30.074
4,1973,2,Commercial,1,0.515
5,1973,2,Electric Power,1,79.331
6,1973,2,Industrial,1,90.320
7,1973,2,Residential,1,27.164


### 3. Chronological Train-Test Split (Out-of-Time Validation)
**Objective:** Split the dataset strictly chronologically without random shuffling to simulate real-world forecasting and prevent temporal look-ahead bias:
- **Training Set (1973–2018):** 46 full years (2,208 observations, ~90.2%).
- **Testing Set (2019–2023):** Unseen 5-year forecast horizon (240 observations, ~9.8%).

In [26]:
SPLIT_YEAR = 2018

train_df = df_feat[df_feat['Year'] <= SPLIT_YEAR].copy()
test_df = df_feat[df_feat['Year'] > SPLIT_YEAR].copy()

assert len(train_df) + len(test_df) == len(df_feat), "Row count mismatch after split!"
assert train_df['Year'].max() < test_df['Year'].min(), "Temporal overlap detected between Train and Test!"
print(f"Train shape: {train_df.shape} | Years: {train_df['Year'].min()}–{train_df['Year'].max()} ({len(train_df)/len(df_feat)*100:.1f}%)")
print(f"Test shape : {test_df.shape}  | Years: {test_df['Year'].min()}–{test_df['Year'].max()} ({len(test_df)/len(df_feat)*100:.1f}%)")

Train shape: (2208, 5) | Years: 1973–2018 (90.2%)
Test shape : (240, 5)  | Years: 2019–2023 (9.8%)


In [27]:
df_feat.head()

,Year,Month,Sector,Total Renewable Energy,t
0,1973,1,Commercial,0.570,0
1,1973,1,Electric Power,89.223,0
2,1973,1,Industrial,99.973,0
3,1973,1,Residential,30.074,0
4,1973,2,Commercial,0.515,1


### 4. Feature Encoding & Trend Interaction Terms
**Objective:** Transform categorical time/domain features into a regression-ready numeric design matrix (`18` features total) while preventing the **Dummy Variable Trap** (perfect multicollinearity):
- **`Sector` Dummies (`drop_first=True`):** Encodes `Electric Power`, `Industrial`, and `Residential` relative to `Commercial` (reference baseline).
- **`Sector * t` Interaction Terms:** Multiplies each `Sector` dummy by `t` so OLS can learn sector-specific growth slopes (as observed in EDA Plot 3) rather than forcing parallel trends.
- **`Month` Dummies (`drop_first=True`):** Encodes Months `2–12` relative to January (`Month = 1`) to capture non-linear seasonal cycles flexibly.

In [ ]:
sector_cats = sorted(train_df['Sector'].unique())

print(sector_cats)
month_cats = list(range(1,13))

def build_feature_matrix(df_split, sector_cat, month_cat):
    """Build leakage-free OLS feature matrix (X) & target (Y)"""
    df_t = df_split[['t']].copy()

    sector_series = pd.Series(
        pd.Categorical(df_split['Sector'], categories=sector_cat),
        index=df_split.index
    )

    month_series = pd.Series(
        pd.Categorical(df_split['Month'], categories=month_cat),
        index=df_split.index
    )

    sector_dummies = pd.get_dummies(sector_series, prefix='Sector', drop_first=True, dtype=int)
    month_dummies = pd.get_dummies(month_series, prefix='Month', drop_first=True, dtype=int)

    sector_t = sector_dummies.multiply(df_split['t'],axis=0)
    sector_t.columns = [f"{col}_x_t" for col in sector_dummies.columns]

    X = pd.concat([df_t, sector_dummies, sector_t, month_dummies], axis=1)
    y = df_split['Total Renewable Energy'].copy()

    return X,y


['Commercial', 'Electric Power', 'Industrial', 'Residential']


In [44]:
X_train, y_train = build_feature_matrix(train_df, sector_cats, month_cats)
X_test, y_test = build_feature_matrix(test_df, sector_cats, month_cats)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((2208, 18), (2208,), (240, 18), (240,))

In [42]:
X_train.head()

,t,Sector_Electric Power,Sector_Industrial,Sector_Residential,Sector_Electric Power_x_t,Sector_Industrial_x_t,Sector_Residential_x_t,Month_2,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9,Month_10,Month_11,Month_12
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


In [41]:
assert list(X_train.columns) == list(X_test.columns), "Train/Test feature mismatch!"
assert not X_train.isna().any().any() and not X_test.isna().any().any(), "Unexpected NaNs!"


### 5. Feature Scaling & Dataset Export
**Objective:** Standardize continuous trend features (`t` and `Sector * t` interactions) using `StandardScaler` fit **strictly on `X_train`** while keeping binary `0/1` dummy variables intact, then export both unscaled and scaled modeling matrices to `data/processed/`.

In [ ]:
from sklearn.preprocessing import StandardScaler

continuous_cols = [
    't',
    'Sector_Electric Power_x_t',
    'Sector_Industrial_x_t',
    'Sector_Residential_x_t',
]

sc = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train[continuous_cols] = sc.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = sc.transform(X_test[continuous_cols])

assert np.allclose(X_train_scaled[continuous_cols].mean(), 0, atol=1e-9), "Train scaled mean != 0"
assert np.allclose(X_train_scaled[continuous_cols].std(ddof=0), 1, atol=1e-9), "Train scaled std != 1"
assert X_test_scaled['t'].min() > 1.0, "Expected future test 't' to sit strictly above train mean!"


In [51]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

exports = {
    'X_train.csv': X_train,
    'X_test.csv': X_test,
    'X_train_scaled.csv': X_train_scaled,
    'X_test_scaled.csv': X_test_scaled,
    'y_train.csv': y_train.to_frame(),
    'y_test.csv': y_test.to_frame(),
}

for filename, data in exports.items():
    out_path = PROCESSED_DIR / filename
    data.to_csv(out_path, index=False)
    print(f"  - {out_path} ({data.shape[0]:,} rows x {data.shape[1]} cols)")

  - ..\data\processed\X_train.csv (2,208 rows x 18 cols)
  - ..\data\processed\X_test.csv (240 rows x 18 cols)
  - ..\data\processed\X_train_scaled.csv (2,208 rows x 18 cols)
  - ..\data\processed\X_test_scaled.csv (240 rows x 18 cols)
  - ..\data\processed\y_train.csv (2,208 rows x 1 cols)
  - ..\data\processed\y_test.csv (240 rows x 1 cols)
